# DRfold2 \u2014 Google Colab Inference Notebook

**Project:** RNA 3D Structure Prediction for Template-Sparse RNA Sequences  
**Model:** DRfold2

This notebook provides a reproducible Colab workflow for DRfold2.

### Primary benchmark input
DRfold2 receives an **RNA sequence in FASTA format**.

Do **not** provide the experimental native 3D structure during inference. Native structures are used only later for evaluation.

Official repository: `https://github.com/leeyang/DRfold2`

## 1. GPU check

Enable a GPU in Colab using **Runtime \u2192 Change runtime type \u2192 GPU**.

In [ ]:
import os, sys, platform, subprocess, time, threading
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

subprocess.run(["nvidia-smi"], check=False)

## 2. Install dependencies

The official DRfold2 installer downloads model weights and builds Arena.  
`clang` is installed because Arena compilation requires `clang++` on the tested Ubuntu setup.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq clang wget git build-essential
!pip -q install numpy scipy biopython

## 3. Clone the official DRfold2 repository

In [ ]:
%cd /content

if not os.path.exists("/content/DRfold2"):
    !git clone https://github.com/leeyang/DRfold2.git
else:
    print("Repository already exists; reusing it.")

%cd /content/DRfold2

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("DRfold2 commit:", commit)

## 4. Download model weights and build Arena

The official `install.sh` downloads the model hub and builds Arena.

In [ ]:
%cd /content/DRfold2

arena_ok = os.path.isfile("/content/DRfold2/Arena/Arena")
weights_ok = os.path.isdir("/content/DRfold2/model_hub")

if arena_ok and weights_ok:
    print("Weights and Arena already installed.")
else:
    !bash install.sh

## 5. Verify installation

In [ ]:
checks = {
    "DRfold_infer.py": Path("/content/DRfold2/DRfold_infer.py").exists(),
    "model_hub": Path("/content/DRfold2/model_hub").is_dir(),
    "Arena executable": Path("/content/DRfold2/Arena/Arena").is_file(),
    "bundled test FASTA": Path("/content/DRfold2/test/seq.fasta").is_file(),
}
for name, ok in checks.items():
    print(f"{name:22s}: {'OK' if ok else 'MISSING'}")

## 6. Select input FASTA

Start with the bundled test FASTA. Later you can upload any benchmark target FASTA.

In [ ]:
INPUT_FASTA = "/content/DRfold2/test/seq.fasta"
OUTPUT_DIR = "/content/drfold2_output"

print("Input:", INPUT_FASTA)
print("Output:", OUTPUT_DIR)

with open(INPUT_FASTA) as f:
    print(f.read())

### Optional: upload your own FASTA

In [ ]:
# Uncomment when needed:
#
# from google.colab import files
# uploaded = files.upload()
# uploaded_name = next(iter(uploaded))
# INPUT_FASTA = f"/content/{uploaded_name}"
# OUTPUT_DIR = f"/content/drfold2_output_{Path(uploaded_name).stem}"
# print("Using:", INPUT_FASTA)

## 7. Run inference + measure runtime and observed peak GPU memory

GPU memory is sampled from `nvidia-smi` every 0.5 s during inference.

In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

gpu_samples = []
stop_monitor = False

def monitor_gpu():
    global stop_monitor
    while not stop_monitor:
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                text=True
            ).strip().splitlines()
            if out:
                gpu_samples.append(int(out[0]))
        except Exception:
            pass
        time.sleep(0.5)

t = threading.Thread(target=monitor_gpu, daemon=True)
t.start()

start = time.perf_counter()

proc = subprocess.run(
    [sys.executable, "DRfold_infer.py", INPUT_FASTA, OUTPUT_DIR],
    cwd="/content/DRfold2"
)

elapsed = time.perf_counter() - start
stop_monitor = True
t.join(timeout=2)

peak_gpu_mb = max(gpu_samples) if gpu_samples else None

print("Exit code:", proc.returncode)
print(f"Runtime: {elapsed:.2f} s")
print("Peak observed GPU memory:", f"{peak_gpu_mb} MiB" if peak_gpu_mb else "Unavailable")

## 8. Locate final prediction

The official interface saves final predictions under:

`<output_dir>/relax/model_*.pdb`

In [ ]:
final_pdbs = sorted(Path(OUTPUT_DIR).glob("relax/model_*.pdb"))

if final_pdbs:
    print("Final prediction(s):")
    for p in final_pdbs:
        print(" -", p)
else:
    print("No final relaxed PDB found.")

## 9. Save run metadata

In [ ]:
import csv

commit = subprocess.check_output(
    ["git", "-C", "/content/DRfold2", "rev-parse", "HEAD"],
    text=True
).strip()

record = {
    "model": "DRfold2",
    "model_commit": commit,
    "input_fasta": INPUT_FASTA,
    "output_dir": OUTPUT_DIR,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
    "runtime_seconds": round(elapsed, 3),
    "peak_gpu_memory_mib_observed": peak_gpu_mb,
    "final_pdb": str(final_pdbs[0]) if final_pdbs else ""
}

LOG_PATH = "/content/drfold2_run_log.csv"
with open(LOG_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=record.keys())
    w.writeheader()
    w.writerow(record)

print("Saved:", LOG_PATH)
print(record)

## 10. Optional: save outputs to Google Drive

In [ ]:
# Uncomment when persistent storage is needed:
#
# from google.colab import drive
# import shutil
#
# drive.mount("/content/drive")
# destination = "/content/drive/MyDrive/RNA3D_Benchmark/DRfold2"
# os.makedirs(destination, exist_ok=True)
#
# shutil.copytree(
#     OUTPUT_DIR,
#     os.path.join(destination, Path(OUTPUT_DIR).name),
#     dirs_exist_ok=True
# )
# shutil.copy2(LOG_PATH, destination)
# print("Saved to:", destination)

## 11. Future benchmark workflow

For each selected RNA target:

1. provide its RNA sequence as FASTA
2. run DRfold2 using the same settings
3. save the predicted PDB
4. record runtime, GPU and peak VRAM
5. compare against the native experimental structure only after inference
6. calculate TM-score, RMSD, lDDT and INF in the separate evaluation pipeline

A later batch wrapper can automate this workflow for all benchmark targets.